# MindSpaceOne — Data Preprocessing

This notebook converts the original SoSci Survey numeric export into the cleaned analysis dataset used by `Analysis.ipynb`.

**Input**
- `data/Collection/data_numeric_mindspaceOne_2025-01-07.csv`

**Outputs**
- `data/cleaned_mindspaceone_2025-Jan-30.csv`
- `data/cleaned_mindspaceone_2025-Jan-30.xlsx`


## 1. Setup


In [1]:
required_packages <- c(
  "dplyr", "readr", "janitor", "writexl"
)

missing_packages <- setdiff(required_packages, rownames(installed.packages()))
if (length(missing_packages) > 0) {
  stop(
    "Missing R packages: ", paste(missing_packages, collapse = ", "),
    ". Install them with install.packages() and rerun the notebook."
  )
}

invisible(lapply(required_packages, library, character.only = TRUE))



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘janitor’ was built under R version 4.3.3”

Attaching package: ‘janitor’


The following objects are masked from ‘package:stats’:

    chisq.test, fisher.test


Warning message:
“package ‘writexl’ was built under R version 4.3.3”


In [2]:
# Locate the repository root without relying on a user-specific absolute path.
find_project_root <- function(start = getwd()) {
  current <- normalizePath(start, winslash = "/", mustWork = TRUE)

  repeat {
    is_repo_root <- dir.exists(file.path(current, "data")) &&
      dir.exists(file.path(current, "scr")) &&
      file.exists(file.path(current, "README.md"))

    if (is_repo_root) return(current)

    parent <- dirname(current)
    if (identical(parent, current)) {
      stop("Could not locate the MindSpaceOne repository root. Start Jupyter from the repository or scr/ directory.")
    }
    current <- parent
  }
}

project_root <- find_project_root()
data_dir <- file.path(project_root, "data")
collection_dir <- file.path(data_dir, "Collection")
graphs_dir <- file.path(project_root, "graphs")
dir.create(graphs_dir, recursive = TRUE, showWarnings = FALSE)

cat("Project root:", project_root, "\n")


Project root: /Users/stevenschepanski/Documents/04_ANALYSIS/MindspaceOne 


## 2. Import the original survey export


In [3]:
raw_file <- file.path(
  collection_dir,
  "data_numeric_mindspaceOne_2025-01-07.csv"
)

if (!file.exists(raw_file)) {
  stop("Input file not found: ", raw_file)
}

# The SoSci export is semicolon-delimited UTF-16 and contains one metadata row
# before the actual header.
mind <- readr::read_delim(
  file = raw_file,
  delim = ";",
  locale = readr::locale(encoding = "UTF-16"),
  skip = 1,
  col_types = readr::cols(.default = readr::col_guess()),
  name_repair = "unique",
  show_col_types = FALSE
)

cat("Imported:", nrow(mind), "rows x", ncol(mind), "columns\n")


New names:
• `Personenkennung (SERIAL)` -> `Personenkennung (SERIAL)...25`
• `Personenkennung (SERIAL)` -> `Personenkennung (SERIAL)...80`
• `Personenkennung (SERIAL)` -> `Personenkennung (SERIAL)...120`
• `Personenkennung (SERIAL)` -> `Personenkennung (SERIAL)...121`
Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”


Imported: 8238 rows x 147 columns


In [4]:
# Standardize the long German survey labels to stable snake_case names.
# readr already converts the declared UTF-16 input to R's internal text encoding,
# so a second iconv() pass is neither necessary nor desirable.
mind <- janitor::clean_names(mind)


## 3. Select and rename analysis variables


In [5]:
column_mapping <- c(
  # Identification & Metadata
  "Serial_Nr" = "personenkennung_oder_teilnahmecode_sofern_verwendet",
  "Quest_Nr" = "fragebogen_der_im_interview_verwendet_wurde",
  "Mode" = "interview_modus",
  "time_started" = "zeitpunkt_zu_dem_das_interview_begonnen_hat_europe_berlin",
  "questions_ended" = "wurde_die_befragung_abgeschlossen_letzte_seite_erreicht",
  "viewer" = "hat_der_teilnehmer_den_fragebogen_nur_angesehen_ohne_die_pflichtfragen_zu_beantworten",

  # Participant Behavior & Survey Progress
  "where_did_the_viewer_ended" = "seite_die_der_teilnehmer_zuletzt_bearbeitet_hat",
  "where_did_the_participant_ended" = "letzte_seite_die_im_fragebogen_bearbeitet_wurde",
  "proportion_missing_answers" = "anteil_fehlender_antworten_in_prozent",
  "proportion_missing_answers_weighted" = "anteil_fehlender_antworten_gewichtet_nach_relevanz",
  "answering_speed" = "ausfull_geschwindigkeit_relativ",

  # Study Group Assignment
  "my_group" = "my_group_my_group",
  "group" = "rando_gezogener_code",

  # Participant Demographics & Eligibility
  "participant_age_criteria" = "ausschluss_sind_zwischen_18_und_65_jahre_alt",
  "participant_location_criteria" = "ausschluss_haben_ihren_aktuellen_wohnsitz_in_deutschland",
  "participant_shipping_agreement" = "ausschluss_sind_mit_der_angabe_ihrer_postadresse_einverstanden_fur_den_versand_der_aromasprays",
  "participant_psych_disorder" = "ausschluss_haben_aktuell_oder_in_den_letzten_3_jahren_keine_psychischen_vor_begleiterkrankungen",
  "participant_psych_session" = "ausschluss_befinden_sich_aktuell_oder_in_den_letzten_6_monaten_nicht_in_aktiver_psychotherapie_einzelne_sitzungen_oder_probatorische_sitzungen_bis_5_stuck_sind_davon_ausgeschlossen",
  "participant_hynpo_aroma" = "ausschluss_nehmen_aktuell_nicht_an_einer_ahnlichen_studie_mit_inhalten_zu_hypno_oder_aromatherapie_teil",
  "participation_consent" = "consent",

  # Recruitment Information
  "Serial_Nr_25" = "personenkennung_serial_25",
  "recruit_flyer" = "von_studie_erfahren_flyer_oder_aushang",
  "recruit_kleinanzeigen" = "von_studie_erfahren_kleinanzeigen",
  "recruit_instagram" = "von_studie_erfahren_instagram",
  "recruit_primavera" = "von_studie_erfahren_primavera",
  "recruit_personal" = "von_studie_erfahren_personliche_empfehlung_durch_gesprach",
  "recruit_rest" = "von_studie_erfahren_sonstiges",

  # Demographics
  "gender" = "geschlecht",
  "age_group" = "altersgruppe",
  "education" = "formale_bildung",
  "educational_degree" = "beruflicher_bildungsabschluss",
  "income" = "einkommen",

  # Occupation
  "occupation_student" = "beschaftigung_schuler_in_auszubildende_oder_studierende",
  "occupation_employee" = "beschaftigung_angestellte",
  "occupation_civil_servant" = "beschaftigung_beamtete",
  "occupation_self_employed" = "beschaftigung_selbststandige",
  "occupation_job_less" = "beschaftigung_arbeitslos_arbeit_suchend",
  "occupation_retirement" = "beschaftigung_in_rente_pension",
  "occupation_rest" = "beschaftigung_sonstiges",

  # Stress Factors
  "stressor_minors_0to6" = "stressoren_i_minderjahrige_s_kind_er_von_0_bis_6_jahren",
  "stressor_minor_7to13" = "stressoren_i_minderjahrige_s_kind_er_zwischen_7_und_13_jahren",
  "stressor_minor_14above" = "stressoren_i_jugendliche_s_kind_er_ab_14_jahren",
  "stressor_minor_handicapped" = "stressoren_i_pflegebedurftige_s_kind_er_mit_einer_korperlichen_oder_psychischen_beeintrachtigung",
  "stressor_adult_handicapped" = "stressoren_i_pflegebedurftige_r_erwachsene_r_mit_einer_korperlichen_oder_psychischen_beeintrachtigung",
  "stressor_shifting_shifts" = "stressoren_ii_wechsel_zwischen_verschiedenen_schichten",
  "stressor_night_shift" = "stressoren_ii_nachtschichten",
  "stressor_weekend_work" = "stressoren_ii_wochenendarbeitszeiten_inkl_sonntage_und_feiertage",
  "stressor_seated_work" = "stressoren_iii_ich_arbeite_vorwiegend_im_sitzen",
  "stressor_seated_stood_work" = "stressoren_iii_ich_arbeite_teils_im_sitzen_teils_im_stehen",
  "stressor_physical_work" = "stressoren_iii_ich_arbeite_in_korperlich_anstrengender_arbeit",

  # Relaxation Techniques
  "relaxation_pmr" = "entspannungsmethoden_progressive_muskelentspannung",
  "relaxation_aroma" = "entspannungsmethoden_aromatherapie",
  "relaxation_yoga" = "entspannungsmethoden_yoga",
  "relaxation_meditation" = "entspannungsmethoden_meditation",
  "relaxation_massagen" = "entspannungsmethoden_massagen",
  "relaxation_sport" = "entspannungsmethoden_sport",
  "relaxation_hypnosis" = "entspannungsmethoden_hypnose",
  "relaxation_at" = "entspannungsmethoden_autogenes_training",
  "relaxation_breathing" = "entspannungsmethoden_atemubungen",
  "relaxation_rest" = "entspannungsmethoden_sonstiges",

  # Mbdf questionnaire
  "mdbf_l1_1" = "mdbf_l1_1_zufrieden",
  "mdbf_l1_2" = "mdbf_l1_2_ausgeruht",
  "mdbf_l1_3" = "mdbf_l1_3_ruhelos",
  "mdbf_l1_4" = "mdbf_l1_4_schlecht",
  "mdbf_l1_5" = "mdbf_l1_5_schlapp",
  "mdbf_l1_6" = "mdbf_l1_6_gelassen",
  "mdbf_l1_7" = "mdbf_l1_7_mude",
  "mdbf_l1_8" = "mdbf_l1_8_gut",
  "mdbf_l2_9" = "mdbf_l2_9_unruhig",
  "mdbf_l2_10" = "mdbf_l2_10_munter",
  "mdbf_l2_11" = "mdbf_l2_11_unwohl",
  "mdbf_l2_12" = "mdbf_l2_12_entspannt",
  "mdbf_l2_13" = "mdbf_l2_13_schlafrig",
  "mdbf_l2_14" = "mdbf_l2_14_wohl",
  "mdbf_l2_15" = "mdbf_l2_15_ausgeglichen",
  "mdbf_l2_16" = "mdbf_l2_16_unglucklich",
  "mdbf_l3_17" = "mdbf_l3_17_wach",
  "mdbf_l3_18" = "mdbf_l3_18_unzufrieden",
  "mdbf_l3_19" = "mdbf_l3_19_angespannt",
  "mdbf_l3_20" = "mdbf_l3_20_frisch",
  "mdbf_l3_21" = "mdbf_l3_21_glucklich",
  "mdbf_l3_22" = "mdbf_l3_22_nervos",
  "mdbf_l3_23" = "mdbf_l3_23_ermattet",
  "mdbf_l3_24" = "mdbf_l3_24_ruhig",

  # PSS questionnaire
  "pss1" = "pss_1_wie_oft_hatten_sie_sich_im_letzten_monat_daruber_aufgeregt_dass_etwas_vollig_unerwartetes_eingetreten_ist",
  "pss2" = "pss_2_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_wichtige_dinge_in_ihrem_leben_nicht_beeinflussen_zu_konnen",
  "pss3" = "pss_3_wie_oft_hatten_sie_sich_im_letzten_monat_nervos_und_gestresst_gefuhlt",
  "pss4" = "pss_4_wie_oft_hatten_sie_sich_im_letzten_monat_sicher_im_umgang_mit_personlichen_aufgaben_und_problemen_gefuhlt",
  "pss5" = "pss_5_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_dass_sich_die_dinge_nach_ihren_vorstellungen_entwickeln",
  "pss6" = "pss_6_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_mit_all_den_anstehenden_aufgaben_und_problemen_nicht_richtig_umgehen_zu_konnen",
  "pss7" = "pss_7_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_mit_arger_in_ihrem_leben_klar_zu_kommen",
  "pss8" = "pss_8_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_alles_im_griff_zu_haben",
  "pss9" = "pss_9_wie_oft_hatten_sie_sich_im_letzten_monat_daruber_geargert_wichtige_dinge_nicht_beeinflussen_zu_konnen",
  "pss10" = "pss_10_wie_oft_hatten_sie_im_letzten_monat_das_gefuhl_dass_sich_die_probleme_so_aufgestaut_haben_dass_sie_diese_nicht_mehr_bewaltigen_konnen",

  # WHO5 questionnaire
  "who5_1" = "who_5_war_ich_froh_und_guter_laune",
  "who5_2" = "who_5_habe_ich_mich_ruhig_und_entspannt_gefuhlt",
  "who5_3" = "who_5_habe_ich_mich_energisch_und_aktiv_gefuhlt",
  "who5_4" = "who_5_habe_ich_mich_beim_aufwachen_frisch_und_ausgeruht_gefuhlt",
  "who5_5" = "who_5_war_mein_alltag_voller_dinge_die_mich_interessieren",

  # General questions
  "pre_mood" = "pre1_meine_stimmung_ist",
  "pre_emotions" = "pre3_ich_fuhle_mich",
  "post_mood" = "post1_meine_stimmung_ist",
  "post_emotions" = "post3_ich_fuhle_mich",
  "aroma_received" = "aroma_erhalten",
  "aroma_tested" = "aroma_testen"
)


In [6]:
# Fail early if the source export no longer matches the expected survey schema.
missing_source_columns <- setdiff(unname(column_mapping), names(mind))
if (length(missing_source_columns) > 0) {
  stop(
    "The raw export is missing expected columns: ",
    paste(missing_source_columns, collapse = ", ")
  )
}

mind <- mind %>%
  dplyr::rename(!!!column_mapping) %>%
  dplyr::select(dplyr::all_of(names(column_mapping)))


## 4. Recode categorical variables


In [7]:
mind <- mind %>%
  mutate(
    # Labeling 'questions_ended' (Survey Completion Status)
    questions_ended = factor(
      questions_ended,
      levels = c(0, 1),
      labels = c("not_finished", "finished")
    ),

    # Labeling 'viewer' (Did the participant only view the survey?)
    viewer = factor(
      viewer,
      levels = c(0, 1),
      labels = c("participants", "only_viewer")
    ),

    # Labeling 'participant_age_criteria' (Eligible Age?)
    participant_age_criteria = factor(
      participant_age_criteria,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participant_location_criteria'
    participant_location_criteria = factor(
      participant_location_criteria,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participant_shipping_agreement' (Agreed to provide shipping address?)
    participant_shipping_agreement = factor(
      participant_shipping_agreement,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participant_psych_disorder'
    participant_psych_disorder = factor(
      participant_psych_disorder,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participant_psych_session'
    participant_psych_session = factor(
      participant_psych_session,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participant_hynpo_aroma'
    participant_hynpo_aroma = factor(
      participant_hynpo_aroma,
      levels = c(1, 2),
      labels = c("no", "yes")
    ),

    # Labeling 'participation_consent'
    # The value '-9' means missing data and should be converted to NA.
    participation_consent = factor(
      participation_consent,
      levels = c(1, 2, -9),
      labels = c("no", "yes", NA)
    ),

    # Labeling 'group'
    group = factor(
      group,
      levels = c(1, 2, 3, 4),
      labels = c("Control", "Aroma", "Hypno", "Aroma_Hypno")
    ),

    # Labels for recruitment information
    recruit_flyer = factor(recruit_flyer, levels = c(1, 2), labels = c("no", "yes")),
    recruit_kleinanzeigen = factor(recruit_kleinanzeigen, levels = c(1, 2), labels = c("no", "yes")),
    recruit_instagram = factor(recruit_instagram, levels = c(1, 2), labels = c("no", "yes")),
    recruit_primavera = factor(recruit_primavera, levels = c(1, 2), labels = c("no", "yes")),
    recruit_personal = factor(recruit_personal, levels = c(1, 2), labels = c("no", "yes")),
    recruit_rest = factor(recruit_rest, levels = c(1, 2), labels = c("no", "yes")),

    # Labeling for gender
    gender = factor(
      gender,
      levels = c(1, 2, 3, -9),
      labels = c("female", "male", "divers", NA)
    ),

    # Labeling for age
    age_group = factor(
      age_group,
      levels = c(2, 3, 4, 5, 6, -9),
      labels = c("18-25", "26-35", "36-45", "46-55", "56-65", NA)
    ),

    # Labeling for education
    education = factor(
      education,
      levels = c(1, 2, 3, 4, 7, 9, -9),
      labels = c("pupil", "no degree", "Lower secondary school", "Higher secondary school", "High school", "others", NA)
    ),

    # Labeling for educational degree
    educational_degree = factor(
      educational_degree,
      levels = c(1, 4, 11, 12, -9),
      labels = c("no degree", "apprenticeship", "university degree", "other degree", NA)
    ),

    # Labeling for occupation variables
    occupation_student = factor(occupation_student, levels = c(1, 2), labels = c("no", "yes")),
    occupation_employee = factor(occupation_employee, levels = c(1, 2), labels = c("no", "yes")),
    occupation_civil_servant = factor(occupation_civil_servant, levels = c(1, 2), labels = c("no", "yes")),
    occupation_self_employed = factor(occupation_self_employed, levels = c(1, 2), labels = c("no", "yes")),
    occupation_job_less = factor(occupation_job_less, levels = c(1, 2), labels = c("no", "yes")),
    occupation_retirement = factor(occupation_retirement, levels = c(1, 2), labels = c("no", "yes")),
    occupation_rest = factor(occupation_rest, levels = c(1, 2), labels = c("no", "yes")),

    # Labeling for income
    income = factor(
      income,
      levels = c(1, 4, 5, 6, 7, 8, -1, -9),
      labels = c("no income", "less than 1000", "1000-1499", "1500-1999", "2000-2999", "3000 and more", "I do not want to answer", NA)
    ),

    # Labeling for stressor variables
    stressor_minors_0to6 = factor(stressor_minors_0to6, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_minor_7to13 = factor(stressor_minor_7to13, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_minor_14above = factor(stressor_minor_14above, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_minor_handicapped = factor(stressor_minor_handicapped, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_adult_handicapped = factor(stressor_adult_handicapped, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_shifting_shifts = factor(stressor_shifting_shifts, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_night_shift = factor(stressor_night_shift, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_weekend_work = factor(stressor_weekend_work, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_seated_work = factor(stressor_seated_work, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_seated_stood_work = factor(stressor_seated_stood_work, levels = c(1, 2), labels = c("not chosen", "chosen")),
    stressor_physical_work = factor(stressor_physical_work, levels = c(1, 2), labels = c("not chosen", "chosen")),

    # Labeling for relaxation variables
    relaxation_pmr = factor(relaxation_pmr, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_aroma = factor(relaxation_aroma, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_yoga = factor(relaxation_yoga, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_meditation = factor(relaxation_meditation, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_massagen = factor(relaxation_massagen, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_sport = factor(relaxation_sport, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_hypnosis = factor(relaxation_hypnosis, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_at = factor(relaxation_at, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_breathing = factor(relaxation_breathing, levels = c(1, 2), labels = c("not chosen", "chosen")),
    relaxation_rest = factor(relaxation_rest, levels = c(1, 2), labels = c("not chosen", "chosen"))
  )


## 5. Remove administrative records and empty fields


In [8]:
mind <- mind %>%
  dplyr::mutate(
    Quest_Nr = as.factor(Quest_Nr),
    Serial_Nr = as.factor(Serial_Nr),
    my_group = as.factor(my_group)
  ) %>%
  # Registration/address/administrative questionnaires are not part of the analysis dataset.
  dplyr::filter(!Quest_Nr %in% c("Reg", "Admin", "Adr")) %>%
  dplyr::select(-Mode)

# Remove mapped variables that are completely empty after the row filter.
mind <- janitor::remove_empty(mind, which = "cols")


## 6. Consolidate occupation


In [9]:
mind <- mind %>%
  dplyr::mutate(occupation = case_when(

    # 1. Retired always takes precedence
    occupation_retirement == "yes" ~ "Retired",

    # 2. Jobless is assigned unless the person is also a student
    occupation_job_less == "yes" & occupation_student != "yes" ~ "Jobless",

    # 3. Student is chosen unless a work-related occupation (Employee, Civil Servant, Self-Employed) is selected
    occupation_student == "yes" & !(occupation_employee == "yes" | 
                                    occupation_civil_servant == "yes" | 
                                    occupation_self_employed == "yes") ~ "Student",

    # 4. Work-related occupations: If multiple selected, the priority order is:
    #    Employee > Civil Servant > Self-Employed (arbitrary order within this category)
    occupation_employee == "yes" ~ "Employee",
    occupation_civil_servant == "yes" ~ "Civil Servant",
    occupation_self_employed == "yes" ~ "Self-Employed",

    # 5. "Other" is only assigned if no other occupation was selected
    occupation_rest == "yes" ~ "Other",

    # Default to NA if no occupation is marked
    TRUE ~ NA_character_  
  )) %>%

  # Remove the original occupation columns
  # ------------------------------------------
  # - After merging the occupation data, the individual occupation columns are no longer needed.
  # - 'select(-column_names)' drops these columns.
  dplyr::select(-occupation_student, -occupation_employee, -occupation_civil_servant, 
                -occupation_self_employed, -occupation_job_less, 
                -occupation_retirement, -occupation_rest)  

# Convert the new 'occupation' column to a factor
# ---------------------------------------------------
# - Ensures that 'occupation' is stored as a categorical variable.
# - This is useful for:
#   - Data visualization (e.g., bar plots)
#   - Statistical modeling (e.g., regression analysis)
mind$occupation <- as.factor(mind$occupation)


## 7. Quality checks


In [10]:
# These dimensions describe the archived cleaned dataset supplied with the project.
expected_rows <- 6828L
expected_cols <- 83L

cat("Cleaned:", nrow(mind), "rows x", ncol(mind), "columns\n")

if (nrow(mind) != expected_rows || ncol(mind) != expected_cols) {
  warning(
    "Cleaned-data dimensions differ from the archived reference (",
    expected_rows, " x ", expected_cols, "). Review the raw export and preprocessing steps."
  )
}

stopifnot(!"Mode" %in% names(mind))
stopifnot(all(c("Serial_Nr", "Quest_Nr", "my_group", "occupation") %in% names(mind)))


Cleaned: 6828 rows x 83 columns


## 8. Save the cleaned dataset


In [11]:
csv_output <- file.path(data_dir, "cleaned_mindspaceone_2025-Jan-30.csv")
xlsx_output <- file.path(data_dir, "cleaned_mindspaceone_2025-Jan-30.xlsx")

readr::write_csv(mind, csv_output)
writexl::write_xlsx(as.data.frame(mind), xlsx_output)

cat("Saved cleaned data to:\n-", csv_output, "\n-", xlsx_output, "\n")


Saved cleaned data to:
- /Users/stevenschepanski/Documents/04_ANALYSIS/MindspaceOne/data/cleaned_mindspaceone_2025-Jan-30.csv 
- /Users/stevenschepanski/Documents/04_ANALYSIS/MindspaceOne/data/cleaned_mindspaceone_2025-Jan-30.xlsx 
